## Projeto Integrador Parte B – Preparação dos Dados

Nomes: Lucas Souza Santos, Ali Ahmad, Leonardo Tonon

### Sobre a limpeza de dados

In [ ]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
plt.style.use('seaborn-darkgrid')

csv_paths = ['/content/UCI_Credit_Card.csv','UCI_Credit_Card.csv','/workspaces/Projeto-integrador-de-Comunica-o-de-dados/UCI_Credit_Card.csv']
for p in csv_paths:
    if os.path.exists(p):
        df = pd.read_csv(p)
        break
else:
    raise FileNotFoundError('Arquivo UCI_Credit_Card.csv não encontrado nos caminhos previstos.')

df.head()

### Estrutura e tipos

In [ ]:
print('Shape:', df.shape)
print('\nTipos:')
print(df.dtypes)

### Valores ausentes

In [ ]:
print(df.isnull().sum())
print('\nTotal de valores ausentes:', df.isnull().sum().sum())

### Duplicatas

In [ ]:
print('Linhas duplicadas:', df.duplicated().sum())

### Distribuições e categorias (contagens)

In [ ]:
for c in ['SEX','EDUCATION','MARRIAGE']:
    if c in df.columns:
        print(f"\n{c}:\n", df[c].value_counts(dropna=False))

### Faixas etárias

In [ ]:
if 'AGE' in df.columns:
    bins = [0,20,30,40,50,60,70,120]
    labels = ['<20','20-29','30-39','40-49','50-59','60-69','70+']
    df['AGE_GROUP'] = pd.cut(df['AGE'], bins=bins, labels=labels, right=False)
    print(df['AGE_GROUP'].value_counts())

### Estatísticas descritivas (numéricas)

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
describe = df[num_cols].describe().T
if 'median' not in describe.columns:
    describe['median'] = df[num_cols].median()
display(describe[['count','mean','median','std','min','25%','50%','75%','max']])

### Correlação (heatmap)

In [ ]:
corr = df.select_dtypes(include=[np.number]).corr()
plt.figure(figsize=(12,10))
sns.heatmap(corr, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Heatmap de correlação')
plt.show()

### Outliers (IQR) — variáveis financeiras

In [ ]:
fin_cols = [c for c in ['LIMIT_BAL'] + [f'BILL_AMT{i}' for i in range(1,7)] + [f'PAY_AMT{i}' for i in range(1,7)] if c in df.columns]
outliers = {}
for c in fin_cols:
    Q1 = df[c].quantile(0.25)
    Q3 = df[c].quantile(0.75)
    IQR = Q3 - Q1
    low = Q1 - 1.5 * IQR
    high = Q3 + 1.5 * IQR
    out_count = df[(df[c] < low) | (df[c] > high)].shape[0]
    outliers[c] = out_count
print('Outliers (IQR) por coluna:')
for k,v in outliers.items():
    print(f'{k}: {v} ({v/len(df):.2%})')

### Transformação e engenharia de atributos

In [ ]:
df_prep = df.copy()
if 'EDUCATION' in df_prep.columns:
    df_prep['EDUCATION'] = df_prep['EDUCATION'].replace([0,4,5,6],'Outros')
if 'MARRIAGE' in df_prep.columns:
    df_prep['MARRIAGE'] = df_prep['MARRIAGE'].replace({0:'Desconhecido/Outros'})
print('EDUCATION:')
if 'EDUCATION' in df_prep.columns:
    print(df_prep['EDUCATION'].value_counts())
print('\nMARRIAGE:')
if 'MARRIAGE' in df_prep.columns:
    print(df_prep['MARRIAGE'].value_counts())

In [ ]:
bill_cols = [c for c in df_prep.columns if c.startswith('BILL_AMT')]
if 'LIMIT_BAL' in df_prep.columns and bill_cols:
    df_prep['MAX_BILL_AMT'] = df_prep[bill_cols].max(axis=1)
    df_prep['UTILIZATION_RATE'] = df_prep['MAX_BILL_AMT'] / df_prep['LIMIT_BAL']
    df_prep['UTILIZATION_RATE'].replace([np.inf, -np.inf], 0, inplace=True)
    df_prep['UTILIZATION_RATE'].fillna(0, inplace=True)
    display(df_prep['UTILIZATION_RATE'].describe())
else:
    print('UTILIZATION_RATE não criada: LIMIT_BAL ou BILL_AMT ausente')

In [ ]:
pay_cols = [c for c in df_prep.columns if c.startswith('PAY_')]
if pay_cols:
    def avg_positive(row):
        vals = [row[c] for c in pay_cols if pd.notna(row[c])]
        positives = [v for v in vals if v > 0]
        return np.mean(positives) if positives else 0
    df_prep['AVG_PAY_STATUS'] = df_prep.apply(avg_positive, axis=1)
    display(df_prep['AVG_PAY_STATUS'].describe())
else:
    print('AVG_PAY_STATUS não criada: colunas PAY_* ausentes')

### Salvamento do conjunto processado

In [ ]:
out_dir = '/workspaces/Projeto-integrador-de-Comunica-o-de-dados/data/processed'
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir,'UCI_Credit_Card_prepared.csv')
df_prep.to_csv(out_path, index=False)
print('Arquivo salvo:', out_path)

### Resumo das etapas realizadas
- Verificação de estrutura, tipos, nulos e duplicatas
- Identificação de outliers por IQR nas variáveis financeiras
- Agrupamento de categorias ruidosas em EDUCATION e MARRIAGE
- Criação das features MAX_BILL_AMT, UTILIZATION_RATE e AVG_PAY_STATUS
- Exportação do dataset preparado